# 40 — De-Duplicated Author Trajectory DiD (Citations per Year · Median)

## Goal
Visualise how **citations received per year** and **publications per year** evolve for award-winning authors  
relative to their award year (`t = 0`), comparing **Junior** vs **Senior** authors.

---

## ⚠️ Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| **De-duplication** | One row per **unique `author_id`**, keeping the *earliest* award year | Authors on multiple award papers would otherwise be counted multiple times, skewing trajectories |
| **Citation metric** | **Citations received *per year*** (not cumulative) | Shows the *flow* of attention; cumulative confounds early- vs late-career starting levels |
| **Aggregation** | **Median** per relative year × group | Robust to the heavy-tailed distributions typical in bibliometrics |
| **Window capping** | ±5 years; for junior authors the pre-award floor = –career_age | Avoids fabricating zero-data years before the author published anything |
| **Breakdowns** | 1. Overall · 2. Split by **conference** · 3. Split by **award type** | Checks whether the signal is driven by a single venue or award category |

---

## Input
`../data/processed/author_yearly_trajectories.csv` — built in notebook 38 (already de-duped & windowed).

## Output
Plots saved to `../data/processed/`

## 0. Imports & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

os.makedirs('../data/processed', exist_ok=True)

traj = pd.read_csv('../data/processed/author_yearly_trajectories.csv')
print(f'Rows loaded   : {len(traj)}')
print(f'Unique authors: {traj["author_id"].nunique()}')
traj.head()

## 1. Verify de-duplication

> **Reminder:** The de-duplication was already applied in notebook 38 when building `author_yearly_trajectories.csv`.  
> Each `author_id` appears only once (earliest award year kept).  
> This cell confirms the invariant holds.

In [ ]:
# One author_id → one award_year (deduplicated upstream)
author_award_counts = traj.groupby('author_id')['award_year'].nunique()
assert (author_award_counts == 1).all(), 'De-duplication violated — some authors have multiple award years'
print('✓ De-duplication confirmed: each author_id maps to exactly one award_year')
print(f'Junior authors : {traj[traj["is_junior"]].author_id.nunique()}')
print(f'Senior authors : {traj[~traj["is_junior"]].author_id.nunique()}')

## 2. Plot helper

### Why median and not mean?

> Citation counts follow a **heavy-tailed distribution** — a small number of star papers attract the bulk of citations.  
> The mean is pulled strongly by these outliers, making it an unreliable summary of the *typical* author trajectory.  
> The **median** is robust to such outliers and better represents what happens to the median award recipient.

### Why citations *per year* and not cumulative?

> Cumulative citations always increase over time by construction, making it hard to detect whether  
> the *rate* of citation actually changed around the award year.  
> Per-year counts reveal the annual *flow* of attention and make the DiD pattern readable.

In [ ]:
COLORS = {'Junior (<5 yr)': '#E07B39', 'Senior (≥5 yr)': '#3A7EBB'}

def did_plot(df, title, savepath):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

    for metric, ax, ylabel, panel_title in [
        ('works_count',    axes[0], 'Median publications per year',       'Publications per year'),
        ('cited_by_count', axes[1], 'Median citations received per year', 'Citations received per year'),
    ]:
        for group, grp_df in df.groupby('seniority'):
            agg = (
                grp_df.groupby('relative_year')[metric]
                      .median()
                      .reset_index()
                      .sort_values('relative_year')
            )
            ax.plot(
                agg['relative_year'], agg[metric],
                marker='o', markersize=5,
                color=COLORS.get(group, 'gray'),
                label=group
            )

        ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6, label='Award year (t=0)')
        ax.set_xlabel('Years relative to award')
        ax.set_ylabel(ylabel)
        ax.set_title(panel_title)
        ax.legend(fontsize=9)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig(savepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {savepath}')

## 3. Overall DiD — all venues & award types combined

In [ ]:
did_plot(traj, 'DiD Trajectories — All venues & award types', '../data/processed/40_did_overall.png')

## 4. Split by Conference

> **Purpose:** Check whether the overall trajectory pattern is consistent across ICWSM and JCDL,  
> or whether the signal is driven by one venue in particular.

In [ ]:
for conf, grp in traj.groupby('conference'):
    savepath = f"../data/processed/40_did_conf_{conf.lower()}.png"
    did_plot(grp, f'DiD Trajectories — {conf}', savepath)

## 5. Split by Award Type

> **Purpose:** Check whether different award categories (Best Paper, Honourable Mention, etc.)  
> show distinct trajectory patterns.
>
> ⚠️ Award types with **fewer than 5 authors in either group** are skipped to avoid noisy / uninterpretable plots.  
> A summary table of skipped award types is printed below each plot block.

In [ ]:
MIN_AUTHORS = 5
skipped = []

for award_type, grp in traj.groupby('award_type'):
    n_junior = grp[grp['is_junior']]['author_id'].nunique()
    n_senior = grp[~grp['is_junior']]['author_id'].nunique()

    if n_junior < MIN_AUTHORS or n_senior < MIN_AUTHORS:
        skipped.append({'award_type': award_type, 'n_junior': n_junior, 'n_senior': n_senior})
        continue

    safe_name = award_type.lower().replace(' ', '_').replace('/', '_')
    savepath = f"../data/processed/40_did_award_{safe_name}.png"
    did_plot(grp, f'DiD Trajectories — {award_type}', savepath)

if skipped:
    print('\n⚠️  Skipped award types (< 5 authors in at least one seniority group):')
    print(pd.DataFrame(skipped).to_string(index=False))
else:
    print('All award types had sufficient authors — nothing skipped.')

## 6. Coverage summary

> The number of authors contributing at each relative year varies — especially at the extremes  
> (very early pre-award years have few junior authors; very late post-award years may have truncated data).  
> This table is **essential context** for interpreting noisy tails in the trajectory plots.

In [ ]:
coverage = (
    traj.groupby(['relative_year', 'seniority'])['author_id']
        .nunique()
        .unstack(fill_value=0)
        .sort_index()
)
print('Authors contributing at each relative year:')
coverage